# FYTA — Sensor Data Quality & Feature Engineering

This notebook covers:

1. Ingest & explore
2. Data quality
3. Cleaning & normalization
4. Unified per-plant feature table
5. Optional anomaly flag

## 1. Ingest & Explore

Load the sensor, contextual-log, and device-to-plant mapping datasets; parse timestamps robustly; and attach plant metadata to each sensor reading.


In [24]:
import pandas as pd
import numpy as np

sensor_df = pd.read_csv("src/data/fyta_sensor_sample.csv")
context_df = pd.read_csv("src/data/fyta_contextual.csv")
mapping_df = pd.read_csv("src/data/user_plant_device_map.csv")

# Robust sensor timestamp parsing: ISO first, then European day-first format.
raw_ts = sensor_df["timestamp"].astype(str)

sensor_df["timestamp"] = pd.to_datetime(
    raw_ts,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

mask = sensor_df["timestamp"].isna()
sensor_df.loc[mask, "timestamp"] = pd.to_datetime(
    raw_ts[mask],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

context_df["created_at"] = pd.to_datetime(
    context_df["created_at"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

sensor_df = sensor_df.merge(
    mapping_df[["device_id", "user_plant_id", "species"]],
    on="device_id",
    how="left"
)


In [25]:
print("Sensor rows:", len(sensor_df))
print("Devices:", sensor_df["device_id"].nunique())
print("Plants:", sensor_df["user_plant_id"].nunique())
print("Substrates:", sensor_df["substrate_label"].nunique())
print("Date range:", sensor_df["timestamp"].min(), "→", sensor_df["timestamp"].max())

print("Invalid timestamps:", sensor_df["timestamp"].isna().sum())
print("Unmapped sensor rows:", sensor_df["user_plant_id"].isna().sum())

print("Readings per device:")
display(sensor_df.groupby("device_id").size().rename("rows").to_frame())

print("Missing values:")
display(sensor_df.isna().sum().rename("missing").to_frame())

print("Numerical summary:")
display(sensor_df[[
    "soil_moisture_vwc",
    "soil_temp_c",
    "ec_us_cm",
    "light_par",
    "air_humidity_pct"
]].describe().T)

Sensor rows: 15950
Devices: 8
Plants: 6
Substrates: 3
Date range: 2026-06-01 00:00:00 → 2026-06-22 01:45:00
Invalid timestamps: 0
Unmapped sensor rows: 0
Readings per device:


,rows
device_id,
SENS-01,2016
SENS-02,2016
SENS-03,2016
SENS-04,2016
SENS-05,1823
SENS-06,2016
SENS-07,2016
SENS-08,2031


Missing values:


,missing
device_id,0
timestamp,0
substrate_label,0
soil_moisture_vwc,0
soil_temp_c,0
ec_us_cm,0
light_par,0
air_humidity_pct,0
user_plant_id,0
species,0


Numerical summary:


,count,mean,std,min,25%,50%,75%,max
soil_moisture_vwc,15950.0,29.359522,12.551057,-5.43,22.86,26.05,33.56,107.12
soil_temp_c,15950.0,27.161629,16.464319,16.10,18.85,21.77,24.10,78.01
ec_us_cm,15950.0,749.939185,220.635514,556.00,675.00,703.00,735.00,2266.00
light_par,15950.0,102.528489,123.467364,0.00,0.00,11.10,226.70,589.00
air_humidity_pct,15950.0,55.046464,13.176824,27.50,42.70,55.10,67.30,131.00


## 2. Data Quality

### Workflow

Raw sensor data  
↓  
**A. Timestamp & reporting irregularities**  
↓  
**B. Duplicate device/timestamp records**  
↓  
**1. Hard-range validation**  
↓  
**2. Identify SENS-07 temperature unit issue**  
↓  
**3. Correct SENS-07 Fahrenheit → Celsius**  
↓  
**4. Sudden-change detection on corrected data**  
↓  
**5. Cross-reference contextual logs**  
↓  
**6. Final classification**

Final labels: `normal`, `bad_data`, `bad_data_unit_mismatch`, `likely_real_behavior`, `uncertain`.


### 2.A Timestamp and reporting irregularities

Sensors are expected to report every **15 minutes**. Deviations are treated as data-availability or ingestion issues, not plant behavior.


### 2.B Duplicate device/timestamp records

First inspect all duplicate device/timestamp pairs, then remove only **exact duplicate measurements**.


In [26]:
duplicate_rows = sensor_df[
    sensor_df["timestamp"].notna()
    & sensor_df.duplicated(
        subset=["device_id", "timestamp"],
        keep=False
    )
].sort_values(["device_id", "timestamp"])

display(duplicate_rows[[
    "device_id", "timestamp",
    "soil_moisture_vwc", "soil_temp_c",
    "ec_us_cm", "light_par", "air_humidity_pct"
]])

measurement_cols = [
    "device_id", "timestamp",
    "soil_moisture_vwc", "soil_temp_c",
    "ec_us_cm", "light_par", "air_humidity_pct"
]

exact_duplicate_mask = (
    sensor_df["timestamp"].notna()
    & sensor_df.duplicated(subset=measurement_cols, keep="first")
)

print("Duplicate device/timestamp rows:", len(duplicate_rows))
print("Exact redundant rows to remove:", exact_duplicate_mask.sum())

sensor_df = sensor_df.loc[~exact_duplicate_mask].reset_index(drop=True)
print("Rows after exact deduplication:", len(sensor_df))


,device_id,timestamp,soil_moisture_vwc,soil_temp_c,ec_us_cm,light_par,air_humidity_pct
14028,SENS-08,2026-06-02 03:15:00,27.68,17.28,650.0,0.0,64.9
15947,SENS-08,2026-06-02 03:15:00,27.68,17.28,650.0,0.0,64.9
14070,SENS-08,2026-06-02 13:45:00,25.75,24.38,620.0,298.2,41.5
15942,SENS-08,2026-06-02 13:45:00,25.75,24.38,620.0,298.2,41.5
14295,SENS-08,2026-06-04 22:00:00,19.70,20.49,721.0,0.0,48.0
15938,SENS-08,2026-06-04 22:00:00,19.70,20.49,721.0,0.0,48.0
14321,SENS-08,2026-06-05 04:30:00,19.63,17.44,687.0,0.0,69.7
15946,SENS-08,2026-06-05 04:30:00,19.63,17.44,687.0,0.0,69.7
14512,SENS-08,2026-06-07 04:15:00,23.53,17.20,688.0,0.0,68.4
15943,SENS-08,2026-06-07 04:15:00,23.53,17.20,688.0,0.0,68.4


Duplicate device/timestamp rows: 30
Exact redundant rows to remove: 15
Rows after exact deduplication: 15935


### 2.1 Hard-range validation

Values are screened against simple technical/plausibility ranges. These are **data-quality screening limits, not plant-health thresholds**, and should be validated against device specifications and horticultural guidance in production.

| Variable | Screening range |
|---|---:|
| Soil moisture VWC | 0–100 % |
| Soil temperature | 15–60 °C |
| EC | 0–3000 µS/cm |
| Light PAR | 0–2000 |
| Air humidity | 0–100 % |


In [27]:
hard_ranges = {
    "soil_moisture_vwc": (0, 100),
    "soil_temp_c": (15, 60),
    "ec_us_cm": (0, 3000),
    "light_par": (0, 2000),
    "air_humidity_pct": (0, 100)
}

for column, (min_value, max_value) in hard_ranges.items():
    sensor_df[f"{column}_violation"] = (
        (sensor_df[column] < min_value)
        | (sensor_df[column] > max_value)
    )

violation_columns = [f"{column}_violation" for column in hard_ranges]
sensor_df["has_hard_violation"] = sensor_df[violation_columns].any(axis=1)

summary = []
for column, (min_value, max_value) in hard_ranges.items():
    summary.append({
        "variable": column,
        "screening_range": f"{min_value}–{max_value}",
        "violations": int(sensor_df[f"{column}_violation"].sum())
    })

display(pd.DataFrame(summary))
print("Rows with ≥1 hard violation:", sensor_df["has_hard_violation"].sum())

print("Violations by device:")
display(
    sensor_df.loc[sensor_df["has_hard_violation"]]
    .groupby("device_id")[violation_columns]
    .sum()
)

,variable,screening_range,violations
0,soil_moisture_vwc,0–100,61
1,soil_temp_c,15–60,2016
2,ec_us_cm,0–3000,0
3,light_par,0–2000,0
4,air_humidity_pct,0–100,10


Rows with ≥1 hard violation: 2087
Violations by device:


,soil_moisture_vwc_violation,soil_temp_c_violation,ec_us_cm_violation,light_par_violation,air_humidity_pct_violation
device_id,,,,,
SENS-04,49,0,0,0,0
SENS-07,0,2016,0,0,0
SENS-08,12,0,0,0,10


### 2.2 Identify the SENS-07 temperature unit issue

SENS-07 reports roughly **62–78** in a field labelled Celsius. Interpreting those values as Fahrenheit produces plausible indoor temperatures, indicating a systematic unit mismatch rather than genuinely extreme soil temperature.


In [28]:
sens07_temp = sensor_df.loc[
    sensor_df["device_id"] == "SENS-07",
    "soil_temp_c"
]

sens07_summary = pd.Series({
    "raw_min": sens07_temp.min(),
    "raw_max": sens07_temp.max(),
    "raw_mean": sens07_temp.mean(),
    "as_celsius_min": (sens07_temp.min() - 32) * 5 / 9,
    "as_celsius_max": (sens07_temp.max() - 32) * 5 / 9,
    "as_celsius_mean": (sens07_temp.mean() - 32) * 5 / 9,
})

display(sens07_summary.to_frame("value"))


,value
raw_min,61.860000
raw_max,78.010000
raw_mean,69.793889
as_celsius_min,16.588889
as_celsius_max,25.561111
as_celsius_mean,20.996605


### 2.3 Correct SENS-07 Fahrenheit → Celsius

The raw temperature is preserved for auditability. Only the identified SENS-07 unit-mismatch records are converted; downstream change detection uses the corrected Celsius values.

In [29]:
sensor_df["soil_temp_c_original"] = sensor_df["soil_temp_c"]
sensor_df["temperature_corrected"] = False

sens07_mask = (
    (sensor_df["device_id"] == "SENS-07")
    & (sensor_df["soil_temp_c"] > 60)
)

sensor_df.loc[sens07_mask, "soil_temp_c"] = (
    sensor_df.loc[sens07_mask, "soil_temp_c"] - 32
) * 5 / 9

sensor_df.loc[sens07_mask, "temperature_corrected"] = True

print("Corrected SENS-07 records:", sensor_df["temperature_corrected"].sum())
display(sensor_df.loc[
    sensor_df["temperature_corrected"],
    ["device_id", "timestamp", "soil_temp_c_original", "soil_temp_c"]
].head())


Corrected SENS-07 records: 2016


,device_id,timestamp,soil_temp_c_original,soil_temp_c
11903,SENS-07,2026-06-01 02:00:00,65.41,18.561111
11904,SENS-07,2026-06-01 02:15:00,64.99,18.327778
11905,SENS-07,2026-06-01 02:30:00,65.46,18.588889
11906,SENS-07,2026-06-01 02:45:00,64.47,18.038889
11907,SENS-07,2026-06-01 03:00:00,65.03,18.350000


### 2.4 Sudden-change detection on corrected data

Hard ranges catch impossible values; they do not catch unusual but technically valid transitions. The thresholds below therefore act as **screening thresholds** for candidate behavioral anomalies.

The current thresholds are:

| Variable | Sudden-change threshold |
|---|---:|
| Soil moisture VWC | > 10 percentage points |
| Soil temperature | > 3 °C |
| EC | > 150 µS/cm |
| PAR light | > 200 |
| Air humidity | > 20 percentage points |

These thresholds are used only to flag unusual changes between consecutive readings. They are not treated as biological ground truth and should ideally be refined using historical sensor behavior, substrate-specific variability, and horticultural or device-domain expertise.


In [30]:
sensor_df = sensor_df.sort_values(["device_id", "timestamp"]).reset_index(drop=True)

jump_thresholds = {
    "soil_moisture_vwc": 10,
    "soil_temp_c": 3,
    "ec_us_cm": 150,
    "light_par": 200,
    "air_humidity_pct": 20
}

for column in jump_thresholds:
    sensor_df[f"{column}_change"] = sensor_df.groupby("device_id")[column].diff()


def detect_sudden_jump(row):
    jumps = []
    for column, threshold in jump_thresholds.items():
        change = row[f"{column}_change"]
        if pd.notna(change) and abs(change) > threshold:
            jumps.append(column)
    return ", ".join(jumps)


sensor_df["sudden_jump"] = sensor_df.apply(detect_sudden_jump, axis=1)
sensor_df["has_sudden_jump"] = sensor_df["sudden_jump"].ne("")

print("Sudden-change records:", sensor_df["has_sudden_jump"].sum())
display(sensor_df.loc[
    sensor_df["has_sudden_jump"],
    ["device_id", "timestamp", "sudden_jump"]
].head(20))


Sudden-change records: 225


,device_id,timestamp,sudden_jump
28,SENS-01,2026-06-01 07:00:00,ec_us_cm
258,SENS-01,2026-06-03 16:30:00,ec_us_cm
432,SENS-01,2026-06-05 12:00:00,soil_moisture_vwc
500,SENS-01,2026-06-06 05:00:00,ec_us_cm
612,SENS-01,2026-06-07 09:00:00,ec_us_cm
614,SENS-01,2026-06-07 09:30:00,ec_us_cm
801,SENS-01,2026-06-09 08:15:00,ec_us_cm
864,SENS-01,2026-06-10 00:00:00,soil_moisture_vwc
866,SENS-01,2026-06-10 00:30:00,ec_us_cm
913,SENS-01,2026-06-10 12:15:00,ec_us_cm


### 2.5 Cross-reference contextual logs

Technically valid jumps are checked against relevant user logs from the preceding 24 hours. Logs are treated as **supporting evidence, not ground truth**: a missing log does not prove that a sensor reading is wrong.


In [31]:
relevant_logs = {
    "soil_moisture_vwc": ["watering", "repotting"],
    "ec_us_cm": ["watering", "fertilising", "repotting"],
    "light_par": ["light"]
}


def find_context(row):
    if not row["has_sudden_jump"]:
        return None

    variable = row["sudden_jump"].split(", ")[0]
    if variable not in relevant_logs:
        return None

    logs = context_df[
        (context_df["user_plant_id"] == row["user_plant_id"])
        & (context_df["log_type"].isin(relevant_logs[variable]))
        & (context_df["created_at"] <= row["timestamp"])
    ].copy()

    if logs.empty:
        return None

    logs["hours_before"] = (
        row["timestamp"] - logs["created_at"]
    ).dt.total_seconds() / 3600

    logs = logs[(logs["hours_before"] >= 0) & (logs["hours_before"] <= 24)]
    if logs.empty:
        return None

    closest = logs.sort_values("hours_before").iloc[0]
    return (
        f"{closest['log_type']} | "
        f"{closest['created_at']} | "
        f"{closest['text']}"
    )


sensor_df["context"] = sensor_df.apply(find_context, axis=1)

print(
    "Sudden changes with supporting context:",
    (sensor_df["has_sudden_jump"] & sensor_df["context"].notna()).sum()
)


Sudden changes with supporting context: 33


### 2.6 Final classification

Classification is deliberately conservative:

- `bad_data_unit_mismatch` — known SENS-07 Fahrenheit/Celsius issue.
- `bad_data` — other hard-range violations.
- `likely_real_behavior` — technically valid sudden change supported by relevant context.
- `uncertain` — technically valid sudden change without enough supporting evidence.
- `normal` — no issue detected by these checks.

A missing contextual log is **not** treated as evidence of bad data.


In [32]:
def classify_record(row):
    if row["temperature_corrected"]:
        return "bad_data_unit_mismatch"

    if row["has_hard_violation"]:
        return "bad_data"

    if row["has_sudden_jump"] and pd.notna(row["context"]):
        return "likely_real_behavior"

    if row["has_sudden_jump"]:
        return "uncertain"

    return "normal"


sensor_df["classification"] = sensor_df.apply(classify_record, axis=1)

display(
    sensor_df["classification"]
    .value_counts()
    .rename("rows")
    .to_frame()
)


,rows
classification,
normal,13679
bad_data_unit_mismatch,2016
uncertain,142
bad_data,71
likely_real_behavior,27


In [33]:
final_anomalies = sensor_df.loc[
    sensor_df["classification"] != "normal",
    [
        "device_id", "user_plant_id", "species", "timestamp", "substrate_label",
        "soil_moisture_vwc", "soil_temp_c_original", "soil_temp_c",
        "ec_us_cm", "light_par", "air_humidity_pct",
        "has_hard_violation", "temperature_corrected",
        "sudden_jump", "context", "classification"
    ]
].copy()

display(final_anomalies.head(5))


,device_id,user_plant_id,species,timestamp,substrate_label,soil_moisture_vwc,soil_temp_c_original,soil_temp_c,ec_us_cm,light_par,air_humidity_pct,has_hard_violation,temperature_corrected,sudden_jump,context,classification
28,SENS-01,UP-1001,Monstera deliciosa,2026-06-01 07:00:00,potting_soil,40.08,19.42,19.42,782.0,84.8,66.5,False,False,ec_us_cm,NaN,uncertain
258,SENS-01,UP-1001,Monstera deliciosa,2026-06-03 16:30:00,potting_soil,25.29,24.18,24.18,630.0,118.9,37.5,False,False,ec_us_cm,NaN,uncertain
432,SENS-01,UP-1001,Monstera deliciosa,2026-06-05 12:00:00,potting_soil,42.25,23.42,23.42,691.0,327.9,52.6,False,False,soil_moisture_vwc,NaN,uncertain
500,SENS-01,UP-1001,Monstera deliciosa,2026-06-06 05:00:00,potting_soil,35.81,18.51,18.51,587.0,0.0,73.0,False,False,ec_us_cm,NaN,uncertain
612,SENS-01,UP-1001,Monstera deliciosa,2026-06-07 09:00:00,potting_soil,26.72,20.43,20.43,779.0,225.9,61.5,False,False,ec_us_cm,NaN,uncertain


# 3. Cleaning & Normalization

Task 2 produced `sensor_df` with raw measurements, anomaly flags, change features,
and quality classifications.

The goal here is to create a smaller modelling-ready table without deleting useful information.

Workflow:

Task 2 `sensor_df`  
↓  
Correct known units  
↓  
Null impossible individual measurements  
↓  
Preserve uncertain and likely-real events  
↓  
Normalize schema / categories / units  
↓  
Keep essential quality metadata  
↓  
`clean_sensor_df`

In [34]:
# Start from Task 2 output
clean_df = sensor_df.copy()

## 3.1 Correct known units

The SENS-07 Fahrenheit-to-Celsius issue was already corrected in Task 2.

The corrected `soil_temp_c` is used for modelling, while
`soil_temp_c_original` is retained for traceability.

In [35]:
sensor_columns = [
    "soil_moisture_vwc",
    "soil_temp_c",
    "ec_us_cm",
    "light_par",
    "air_humidity_pct"
]

for column in sensor_columns:
    clean_df[f"{column}_clean"] = clean_df[column]

## 3.2 Null impossible individual measurements

Only the invalid measurement is set to `NaN`.

The full row is not deleted because the remaining sensor measurements may still be valid.

In [36]:
clean_df.loc[
    clean_df["soil_moisture_vwc_violation"],
    "soil_moisture_vwc_clean"
] = np.nan

clean_df.loc[
    clean_df["air_humidity_pct_violation"],
    "air_humidity_pct_clean"
] = np.nan

clean_df.loc[
    clean_df["ec_us_cm_violation"],
    "ec_us_cm_clean"
] = np.nan

clean_df.loc[
    clean_df["light_par_violation"],
    "light_par_clean"
] = np.nan

clean_df.loc[
    clean_df["soil_temp_c_violation"]
    & (~clean_df["temperature_corrected"]),
    "soil_temp_c_clean"
] = np.nan

## 3.3 Preserve uncertain and likely-real events

`likely_real_behavior` observations are retained because they may represent genuine plant events.

`uncertain` observations are also retained and flagged rather than deleted.

In [37]:
quality_map = {
    "normal": "usable",
    "likely_real_behavior": "usable",
    "uncertain": "usable_with_caution",
    "bad_data": "partially_invalid",
    "bad_data_unit_mismatch": "usable"
}

clean_df["model_quality"] = (
    clean_df["classification"]
    .map(quality_map)
)

## 3.4 Normalize schema / categories / units

Normalization here is keeping labels and formats consistent.


In [38]:
clean_df["substrate_label"] = (
    clean_df["substrate_label"]
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

clean_df["species"] = (
    clean_df["species"]
    .str.strip()
)

## 3.5 Keep essential quality metadata

The detailed anomaly-analysis columns remain in `sensor_df`.

The modelling-ready table keeps only:

- identifiers
- plant metadata
- timestamp
- cleaned sensor measurements
- compact quality information

In [39]:
model_columns = [
    "device_id",
    "user_plant_id",
    "species",
    "timestamp",
    "substrate_label",

    "soil_moisture_vwc_clean",
    "soil_temp_c_clean",
    "ec_us_cm_clean",
    "light_par_clean",
    "air_humidity_pct_clean",

    "classification",
    "model_quality"
]

clean_sensor_df = clean_df[
    model_columns
].copy()

In [40]:
print("Original shape:", sensor_df.shape)
print("Clean shape:", clean_sensor_df.shape)

display(clean_sensor_df.head())

Original shape: (15935, 27)
Clean shape: (15935, 12)


,device_id,user_plant_id,species,timestamp,substrate_label,soil_moisture_vwc_clean,soil_temp_c_clean,ec_us_cm_clean,light_par_clean,air_humidity_pct_clean,classification,model_quality
0,SENS-01,UP-1001,Monstera deliciosa,2026-06-01 00:00:00,potting_soil,40.93,18.65,733.0,0.0,59.6,normal,usable
1,SENS-01,UP-1001,Monstera deliciosa,2026-06-01 00:15:00,potting_soil,40.68,17.95,694.0,0.0,58.8,normal,usable
2,SENS-01,UP-1001,Monstera deliciosa,2026-06-01 00:30:00,potting_soil,40.63,18.52,671.0,0.0,63.6,normal,usable
3,SENS-01,UP-1001,Monstera deliciosa,2026-06-01 00:45:00,potting_soil,40.29,18.47,684.0,0.0,64.4,normal,usable
4,SENS-01,UP-1001,Monstera deliciosa,2026-06-01 01:00:00,potting_soil,40.29,17.19,652.0,0.0,64.0,normal,usable


In [41]:
print("Missing values in cleaned data:")

display(
    clean_sensor_df.isna().sum()
)

Missing values in cleaned data:


device_id                   0
user_plant_id               0
species                     0
timestamp                   0
substrate_label             0
soil_moisture_vwc_clean    61
soil_temp_c_clean           0
ec_us_cm_clean              0
light_par_clean             0
air_humidity_pct_clean     10
classification              0
model_quality               0
dtype: int64

# 4. Unified Per-Plant Feature Table

The cleaned data is sensor-level.  
To create one row per `user_plant_id + timestamp`, measurements from multiple sensors on the same plant are averaged at each timestamp.

This provides a simple plant-level feature table while preserving the original sensor-level data separately.

In [42]:
plant_feature_columns = [
    "soil_moisture_vwc_clean",
    "soil_temp_c_clean",
    "ec_us_cm_clean",
    "light_par_clean",
    "air_humidity_pct_clean"
]

## 4.1 Aggregate sensor readings to plant level

For each plant and timestamp, the cleaned measurements are averaged across available sensors.

For plants with only one sensor, the value remains unchanged.

In [43]:
plant_features_df = (
    clean_sensor_df
    .groupby(
        [
            "user_plant_id",
            "species",
            "timestamp",
            "substrate_label"
        ],
        as_index=False
    )[plant_feature_columns]
    .mean()
)

In [44]:
print("Sensor-level rows:", len(clean_sensor_df))
print("Plant-level rows:", len(plant_features_df))

display(plant_features_df.head())


plant_features_df.to_csv(
    "src/data/plant_feature_table.csv",
    index=False
)

Sensor-level rows: 15935
Plant-level rows: 11903


,user_plant_id,species,timestamp,substrate_label,soil_moisture_vwc_clean,soil_temp_c_clean,ec_us_cm_clean,light_par_clean,air_humidity_pct_clean
0,UP-1001,Monstera deliciosa,2026-06-01 00:00:00,potting_soil,39.895,18.475,688.5,0.0,59.60
1,UP-1001,Monstera deliciosa,2026-06-01 00:15:00,potting_soil,39.970,17.810,668.0,0.0,59.75
2,UP-1001,Monstera deliciosa,2026-06-01 00:30:00,potting_soil,39.815,18.650,669.5,0.0,60.80
3,UP-1001,Monstera deliciosa,2026-06-01 00:45:00,potting_soil,39.610,18.265,679.0,0.0,63.95
4,UP-1001,Monstera deliciosa,2026-06-01 01:00:00,potting_soil,39.630,17.230,681.5,0.0,64.05


# 5. Optional Feature

## 5.1 Environmental Anomaly Count

`environmental_anomaly_count` is a simple plant-specific anomaly feature.

Each plant is compared with its own historical sensor distribution. For moisture, soil temperature, PAR, and EC, a reading is flagged when it falls outside that plant's `Q25–Q75` range.

The final feature counts how many sensor variables are unusual at the same timestamp:

$$
\text{environmental\_anomaly\_count}
=
A_M + A_T + A_L + A_E
$$

where each \(A\) is either:

- `0` → inside the plant's usual range
- `1` → outside the plant's usual range

The final score ranges from:

- `0` → all four measurements are within the plant's usual range
- `1–2` → some environmental variables are unusual
- `3–4` → several variables are unusual simultaneously

This is an anomaly-monitoring signal, not a direct diagnosis of plant stress.


In [45]:
# Sensor variables used for the anomaly count
anomaly_features = [
    "soil_moisture_vwc_clean",
    "soil_temp_c_clean",
    "light_par_clean",
    "ec_us_cm_clean"
]

# Calculate plant-specific Q25 and Q75 thresholds
for feature in anomaly_features:
    q25 = (
        plant_features_df
        .groupby("user_plant_id")[feature]
        .transform(lambda x: x.quantile(0.25))
    )

    q75 = (
        plant_features_df
        .groupby("user_plant_id")[feature]
        .transform(lambda x: x.quantile(0.75))
    )

    # PAR = 0 is expected at night, so do not flag darkness as an anomaly.
    if feature == "light_par_clean":
        anomaly = (
            (plant_features_df[feature] > 0)
            & (
                (plant_features_df[feature] < q25)
                | (plant_features_df[feature] > q75)
            )
        )
    else:
        anomaly = (
            (plant_features_df[feature] < q25)
            | (plant_features_df[feature] > q75)
        )

    plant_features_df[f"{feature}_anomaly"] = anomaly.astype(int)


# Count how many environmental variables are unusual at each timestamp
anomaly_columns = [
    f"{feature}_anomaly"
    for feature in anomaly_features
]

plant_features_df["environmental_anomaly_count"] = (
    plant_features_df[anomaly_columns]
    .sum(axis=1)
)

display(
    plant_features_df[
        [
            "user_plant_id",
            "species",
            "timestamp",
            *anomaly_columns,
            "environmental_anomaly_count"
        ]
    ].head(20)
)


,user_plant_id,species,timestamp,soil_moisture_vwc_clean_anomaly,soil_temp_c_clean_anomaly,light_par_clean_anomaly,ec_us_cm_clean_anomaly,environmental_anomaly_count
0,UP-1001,Monstera deliciosa,2026-06-01 00:00:00,1,1,0,0,2
1,UP-1001,Monstera deliciosa,2026-06-01 00:15:00,1,1,0,1,3
2,UP-1001,Monstera deliciosa,2026-06-01 00:30:00,1,0,0,1,2
3,UP-1001,Monstera deliciosa,2026-06-01 00:45:00,1,1,0,1,3
4,UP-1001,Monstera deliciosa,2026-06-01 01:00:00,1,1,0,1,3
5,UP-1001,Monstera deliciosa,2026-06-01 01:15:00,1,1,0,1,3
6,UP-1001,Monstera deliciosa,2026-06-01 01:30:00,1,1,0,0,2
7,UP-1001,Monstera deliciosa,2026-06-01 01:45:00,1,1,0,1,3
8,UP-1001,Monstera deliciosa,2026-06-01 02:00:00,1,1,0,0,2
9,UP-1001,Monstera deliciosa,2026-06-01 02:15:00,1,1,0,0,2


### Interpretation

A high `environmental_anomaly_count` means several environmental measurements are outside that plant's usual range at the same time.


In [46]:
print("Environmental anomaly count distribution:")
display(
    plant_features_df["environmental_anomaly_count"]
    .value_counts()
    .sort_index()
    .rename_axis("environmental_anomaly_count")
    .reset_index(name="rows")
)

# Save the enriched plant-level feature table
plant_features_df.to_csv(
    "src/data/plant_feature_table.csv",
    index=False
)

Environmental anomaly count distribution:


,environmental_anomaly_count,rows
0,0,1127
1,1,3766
2,2,4440
3,3,2211
4,4,359
